In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Define category URLs to scrape (scrapes >= 60 books in total)
categories = {
    "Travel": "http://books.toscrape.com/catalogue/category/books/travel_2/index.html",
    "Mystery": "http://books.toscrape.com/catalogue/category/books/mystery_3/index.html",
    "Historical Fiction": "http://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html"
}

scraped_books = []

# Loop through each category and scrape book details
for category_name, url in categories.items():
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    books = soup.find_all('article', class_='product_pod')
    
    for book in books:
        title = book.h3.a['title']
        price = book.find('p', class_='price_color').text
        star_rating = book.find('p', class_='star-rating')['class'][1]  # e.g., 'Three'
        availability = book.find('p', class_='instock availability').text.strip()
        
        scraped_books.append({
            'title': title,
            'price': price,
            'star_rating': star_rating,
            'availability': availability,
            'category': category_name
        })

# Create DataFrame from scraped data
df = pd.DataFrame(scraped_books)

# Verify the output
print(f"Total books scraped: {len(df)}")
print(df.head())

Total books scraped: 51
                                               title    price star_rating  \
0                            It's Only the Himalayas  Â£45.17         Two   
1  Full Moon over Noahâs Ark: An Odyssey to Mou...  Â£49.43        Four   
2  See America: A Celebration of Our National Par...  Â£48.87       Three   
3  Vagabonding: An Uncommon Guide to the Art of L...  Â£36.94         Two   
4                               Under the Tuscan Sun  Â£37.33       Three   

  availability category  
0     In stock   Travel  
1     In stock   Travel  
2     In stock   Travel  
3     In stock   Travel  
4     In stock   Travel  


In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Task 1: Scraping (Added 4th category to get >= 60 books)
categories = {
    "Travel": "http://books.toscrape.com/catalogue/category/books/travel_2/index.html",
    "Mystery": "http://books.toscrape.com/catalogue/category/books/mystery_3/index.html",
    "Historical Fiction": "http://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html",
    "Sequential Art": "http://books.toscrape.com/catalogue/category/books/sequential-art_5/index.html"
}

scraped_books = []

for category_name, url in categories.items():
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    books = soup.find_all('article', class_='product_pod')
    
    for book in books:
        title = book.h3.a['title']
        price = book.find('p', class_='price_color').text
        star_rating = book.find('p', class_='star-rating')['class'][1]
        availability = book.find('p', class_='instock availability').text.strip()
        
        scraped_books.append({
            'title': title,
            'price': price,
            'star_rating': star_rating,
            'availability': availability,
            'category': category_name
        })

df = pd.DataFrame(scraped_books)

# Task 2: Data Cleaning
# Clean price to float column (price_gbp)
df['price_gbp'] = df['price'].str.replace('Â£', '').str.replace('£', '').astype(float)

# Convert text rating to integer (1-5)
rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
df['rating'] = df['star_rating'].map(rating_map)

# Parse availability to boolean/integer (1 for in stock, 0 for out)
df['in_stock'] = df['availability'].str.contains('In stock', case=False).astype(int)

# Task 3: Currency Conversion (1 GBP = 105.50 INR)
df['price_inr'] = (df['price_gbp'] * 105.50).round(2)

# Select and organize final clean columns
df = df[['title', 'category', 'price_gbp', 'price_inr', 'rating', 'in_stock']]

# Verify Task 1, 2, and 3
print(f"Total books scraped: {len(df)}")
print(df.head())
print("\nData Types:")
print(df.dtypes)

Total books scraped: 71
                                               title category  price_gbp  \
0                            It's Only the Himalayas   Travel      45.17   
1  Full Moon over Noahâs Ark: An Odyssey to Mou...   Travel      49.43   
2  See America: A Celebration of Our National Par...   Travel      48.87   
3  Vagabonding: An Uncommon Guide to the Art of L...   Travel      36.94   
4                               Under the Tuscan Sun   Travel      37.33   

   price_inr  rating  in_stock  
0    4765.44       2         1  
1    5214.86       4         1  
2    5155.78       3         1  
3    3897.17       2         1  
4    3938.31       3         1  

Data Types:
title            str
category         str
price_gbp    float64
price_inr    float64
rating         int64
in_stock       int64
dtype: object


In [3]:
import sqlite3

# -------------------------------------------------------------
# Task 4: SQLite Schema Creation & Database Loading
# -------------------------------------------------------------
conn = sqlite3.connect('books_data.db')
cursor = conn.cursor()

cursor.execute("PRAGMA foreign_keys = ON;")

# Create categories table
cursor.execute('''
CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT UNIQUE NOT NULL
);
''')

# Create books table with Foreign Key
cursor.execute('''
CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    price_gbp REAL NOT NULL,
    price_inr REAL NOT NULL,
    rating INTEGER NOT NULL,
    in_stock INTEGER NOT NULL,
    category_id INTEGER,
    FOREIGN KEY (category_id) REFERENCES categories (category_id)
);
''')

# Insert Unique Categories
for cat in df['category'].unique():
    cursor.execute("INSERT OR IGNORE INTO categories (category_name) VALUES (?)", (cat,))

conn.commit()

# Map category_name to category_id
category_df = pd.read_sql("SELECT * FROM categories", conn)
category_map = dict(zip(category_df['category_name'], category_df['category_id']))
df['category_id'] = df['category'].map(category_map)

# Insert Books Data
for _, row in df.iterrows():
    cursor.execute('''
    INSERT INTO books (title, price_gbp, price_inr, rating, in_stock, category_id)
    VALUES (?, ?, ?, ?, ?, ?)
    ''', (row['title'], row['price_gbp'], row['price_inr'], row['rating'], row['in_stock'], row['category_id']))

conn.commit()

# -------------------------------------------------------------
# Task 5: Execute 5 Required SQL Queries
# -------------------------------------------------------------
print("=== Query 1: Top 5 Expensive Books in INR (ORDER BY, LIMIT) ===")
print(pd.read_sql("SELECT title, price_inr FROM books ORDER BY price_inr DESC LIMIT 5;", conn))

print("\n=== Query 2: Books with 5-Star Rating (SELECT, WHERE) ===")
print(pd.read_sql("SELECT title, rating, price_gbp FROM books WHERE rating = 5;", conn))

print("\n=== Query 3: Distinct Categories (DISTINCT) ===")
print(pd.read_sql("SELECT DISTINCT category_name FROM categories;", conn))

print("\n=== Query 4: Books Priced Between 20 and 40 GBP (BETWEEN, AND) ===")
print(pd.read_sql("SELECT title, price_gbp FROM books WHERE price_gbp BETWEEN 20 AND 40;", conn))

print("\n=== Query 5: SQL JOIN Books with Categories ===")
sql_join_query = """
SELECT b.title, c.category_name, b.price_inr, b.rating 
FROM books b
JOIN categories c ON b.category_id = c.category_id
WHERE b.rating >= 4
ORDER BY b.price_inr DESC;
"""
df_sql_join = pd.read_sql(sql_join_query, conn)
print(df_sql_join.head())

# -------------------------------------------------------------
# Task 6: Reproduce JOIN using Pandas (pd.merge) & Compare
# -------------------------------------------------------------
print("\n=== Task 6: Pandas pd.merge Output ===")
books_df = pd.read_sql("SELECT * FROM books", conn)
cats_df = pd.read_sql("SELECT * FROM categories", conn)

pandas_merged = pd.merge(books_df, cats_df, on='category_id')
pandas_merged = pandas_merged[pandas_merged['rating'] >= 4][['title', 'category_name', 'price_inr', 'rating']]
pandas_merged = pandas_merged.sort_values(by='price_inr', ascending=False).reset_index(drop=True)

print(pandas_merged.head())

# Verification Check
match = df_sql_join.reset_index(drop=True).equals(pandas_merged)
print(f"\nDo SQL JOIN and Pandas Merge match exactly?: {match}")

conn.close()

=== Query 1: Top 5 Expensive Books in INR (ORDER BY, LIMIT) ===
                                     title  price_inr
0            Boar Island (Anna Pigeon #19)    6275.14
1         A Year in Provence (Provence #1)    6000.84
2                      The Past Never Ends    5960.75
3         The Last Painting of Sara de Vos    5860.52
4  A Flight of Arrows (The Pathfinders #2)    5858.42

=== Query 2: Books with 5-Star Rating (SELECT, WHERE) ===
                                                title  rating  price_gbp
0                  1,000 Places to See Before You Die       5      26.08
1              A Time of Torment (Charlie Parker #14)       5      48.35
2   What Happened on Beale Street (Secrets of the ...       5      25.37
3   The Bachelor Girl's Guide to Murder (Herringfo...       5      52.30
4             A Flight of Arrows (The Pathfinders #2)       5      55.53
5                                        Mrs. Houdini       5      30.25
6                               The Passio